# Elliptical power-law lens and Sérsic source

This tutorial builds the first fully elliptical analytic model in Project Akira. Positions and radii are in arcseconds, `q=b/a`, and `phi` is measured anticlockwise from positive x in radians. The public power-law `slope` is the **3D density slope** $\gamma$; internally the projected Tessore--Metcalf slope is $t=\gamma-1$.

In [ ]:
import matplotlib.pyplot as plt
import numpy as np
from akira import (
    CartesianGrid, CompositeLight, EPLSersicParameters,
    EllipticalPowerLaw, ExternalShear, LensPlane, SIS, Sersic,
    fit_epl_sersic_parameters, gaussian_psf, render, simulate_imaging,
)

## Compose and render the model

A `LensPlane` adds the deflections of all its mass components; `CompositeLight` similarly adds source-light components. This gives us an elliptical power law plus external shear without a special high-level model class.

In [ ]:
grid = CartesianGrid(shape=(90, 90), pixel_scale=0.045, supersampling=2)
lens_mass = EllipticalPowerLaw(0.95, slope=2.10, q=0.74, phi=0.30)
shear = ExternalShear(0.04, phi=0.65)
source_light = Sersic(
    intensity=0.12, effective_radius=0.17, sersic_index=1.3,
    centre=(0.075, -0.05), q=0.70, phi=-0.20,
)
lens = LensPlane((lens_mass, shear))
source = CompositeLight((source_light,))
image = render(grid, lens, source)
half_width = grid.shape[1] * grid.pixel_scale / 2
half_height = grid.shape[0] * grid.pixel_scale / 2
extent = (-half_width, half_width, -half_height, half_height)
plt.figure(figsize=(6, 5))
plt.imshow(np.asarray(image), origin='lower', extent=extent, cmap='magma')
plt.xlabel('x [arcsec]'); plt.ylabel('y [arcsec]')
plt.colorbar(label='surface brightness')
plt.title('EPL + shear lens, Sérsic source'); plt.show()

## Verify the SIS limiting case

For `q=1` and $\gamma=2$, the elliptical power law must reduce to the already verified SIS deflection.

In [ ]:
coordinates = grid.coordinates()
epl_limit = EllipticalPowerLaw(0.95, slope=2.0, q=1.0)
sis = SIS(0.95)
maximum_difference = float(np.max(np.abs(
    np.asarray(epl_limit.deflection(coordinates) - sis.deflection(coordinates))
)))
assert maximum_difference < 1e-12
print(f'Maximum EPL/SIS deflection difference: {maximum_difference:.3e} arcsec')

## Simulate an observation and recover four parameters

We add a PSF and seeded Gaussian noise, then fit the Einstein radius, 3D density slope, and the two source-centre coordinates. The remaining shape parameters are deliberately fixed for this controlled architecture test.

In [ ]:
observation = simulate_imaging(
    image, gaussian_psf(0.9, size=7), noise_sigma=0.004,
    pixel_scale=grid.pixel_scale, seed=2026,
)
result = fit_epl_sersic_parameters(
    observation, grid, lens_template=lens_mass, source_template=source_light,
    shear=shear, initial=EPLSersicParameters(0.85, 1.95, 0.02, -0.01),
    bounds=((0.7, 1.2), (1.7, 2.5), (-0.2, 0.2), (-0.2, 0.2)),
)
truth = np.array([0.95, 2.10, 0.075, -0.05])
recovered = np.array([
    result.parameters.einstein_radius, result.parameters.slope,
    result.parameters.source_x, result.parameters.source_y,
])
print('parameter          true       recovered')
for name, true, fitted in zip(
    ('Einstein radius', '3D slope', 'source x', 'source y'), truth, recovered
):
    print(f'{name:16s} {true:9.5f}  {fitted:9.5f}')
print(f'optimizer success: {result.success}; evaluations: {result.evaluations}')

In [ ]:
data = np.asarray(observation.data)
model = np.asarray(result.model_image)
residual_sigma = (data - model) / np.asarray(observation.noise_map)
fig, axes = plt.subplots(1, 3, figsize=(14, 4))
panels = ((data, 'data', 'magma'), (model, 'fitted model', 'magma'),
          (residual_sigma, 'residual / noise', 'coolwarm'))
for axis, (values, title, cmap) in zip(axes, panels):
    shown = axis.imshow(values, origin='lower', extent=extent, cmap=cmap)
    axis.set_title(title); axis.set_xlabel('x [arcsec]')
    fig.colorbar(shown, ax=axis, fraction=0.046)
axes[0].set_ylabel('y [arcsec]')
fig.tight_layout(); plt.show()

The fit above is intentionally not yet a general inference framework. It verifies that the new analytic profiles compose, remain differentiable through JAX, pass through the imaging likelihood, and support simultaneous recovery of several scientifically meaningful parameters.